In [1]:
import geopandas as gpd
import pandas as pd

# Adaptez le chemin à votre fichier
# Le shapefile BRGM s'appelle souvent quelque chose comme :
# "BDGEOFRANCE_1M_SHP/FORMATION_GEOLOGIQUE.shp"
# ou "GEO050K_LION/GEO050K_LION_S_GEOL.shp"

shp_path = r"H:\PFE Loice\Notebooks\Loice_Canc-air_2025\loice_pneumodetect\Data\cartes_geologiques\GEO050K_HARM_075_077_078_091_092_093_094_095\GEO050K_HARM_IDF_S_FGEOL_2154.shp"  # ← à modifier

bd_geol = gpd.read_file(shp_path)

# Vue d'ensemble
print("=== DIMENSIONS ===")
print(f"Nombre de polygones : {len(bd_geol)}")
print(f"Projection : {bd_geol.crs}")

print("\n=== COLONNES DISPONIBLES ===")
print(bd_geol.columns.tolist())

print("\n=== APERÇU DES 5 PREMIÈRES LIGNES ===")
print(bd_geol.head())

=== DIMENSIONS ===
Nombre de polygones : 8495
Projection : PROJCS["RGF93 Lambert 93",GEOGCS["RGF93 geographiques (dms)",DATUM["Reseau_Geodesique_Francais_1993_v1",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6171"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["IGNF","RGF93G"]],PROJECTION["Lambert_Conformal_Conic_2SP"],PARAMETER["latitude_of_origin",46.5],PARAMETER["central_meridian",3],PARAMETER["standard_parallel_1",44],PARAMETER["standard_parallel_2",49],PARAMETER["false_easting",700000],PARAMETER["false_northing",6600000],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["IGNF","LAMB93"]]

=== COLONNES DISPONIBLES ===
['MI_PRINX', 'CARTE', 'CODE', 'CODE_LEG', 'NOTATION', 'DESCR', 'C_FOND', 'M_FOND', 'J_FOND', 'N_FOND', 'NOM_SYMB', 'C_SYMB', 'M_SYMB', 'J_SYMB', 'N_SYMB', 'ROT_SYMB', 'geometry']

=== APERÇU DES 5 PREMIÈRES LIGNES

In [2]:
# Afficher les valeurs uniques des colonnes qui semblent contenir
# la description géologique (les noms varient selon la version BRGM)

colonnes_candidates = [c for c in bd_geol.columns 
                       if any(mot in c.upper() for mot in 
                              ['LITH', 'GEOL', 'NOTA', 'DESC', 'TYPE', 'FORM'])]

print("=== COLONNES PROBABLEMENT UTILES ===")
for col in colonnes_candidates:
    n_uniques = bd_geol[col].nunique()
    exemples = bd_geol[col].dropna().unique()[:5]
    print(f"\n{col} ({n_uniques} valeurs uniques)")
    print(f"  Exemples : {exemples}")

=== COLONNES PROBABLEMENT UTILES ===

NOTATION (92 valeurs uniques)
  Exemples : ['X' 'CF' 'CE' 'D' 'e3C']

DESCR (92 valeurs uniques)
  Exemples : ['Dépôts anthropiques, remblais'
 'Colluvions de versant et de fond de vallon'
 'Colluvions polygéniques, éboulis'
 'Sables éoliens (Etampes). Dunes quaternaires (Dourdan)'
 'Poudingue de Coye']


In [3]:
# Voir les 92 paires NOTATION → DESCR
paires = bd_geol[['NOTATION', 'DESCR']].drop_duplicates().sort_values('NOTATION')
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_colwidth', 80)
print(paires.to_string(index=False))

 NOTATION                                                                                                                               DESCR
       C2                                                                                                Craie marneuse à Inoceramus labiatus
      C4M                                                                                                       Craie à Micraster coranguinum
       C5                                                                                                               Craie  à Belemnitella
  C5Cr-BE                                                                                                Craie blanche à silex à Belemnitella
   C5CrAq                                                                                      Craie blanche à silex, à Actinocamax quadratus
       CE                                                                                                    Colluvions polygéniques, éboulis
      

In [4]:
import geopandas as gpd
import pandas as pd

# ============================================================
# TABLE URANIUM — adaptée à votre BD Géol (Bassin parisien)
# Sources : Wedepohl 1995, Taylor & McLennan 1985,
#           données géochimiques BRGM Bassin parisien
# Unité : ppm U équivalent
# ============================================================

URANIUM_PPM = {

    # --- CRAIES (faible, ~1.5-2.0 ppm) ---
    'C2':       1.8,   # Craie marneuse à Inoceramus
    'C4M':      1.6,   # Craie à Micraster
    'C5':       1.5,   # Craie à Belemnitella
    'C5Cr-BE':  1.5,   # Craie blanche à silex
    'C5CrAq':   1.5,   # Craie blanche à silex (Actinocamax)

    # --- ALLUVIONS (très faible, dilution fluviatile) ---
    'FO':       1.2,   # Alluvions Orvanne
    'FT':       1.3,   # Alluvions très anciennes (>80m)
    'Ft':       1.3,   # Alluvions anciennes (65-80m)
    'Fu':       1.2,   # Alluvions anciennes (65m)
    'Fv':       1.3,   # Cailloutis de Sénart (45-55m)
    'Fw':       1.3,   # Haute terrasse (20-30m)
    'Fx':       1.3,   # Moyenne terrasse (10-20m)
    'Fx-y':     1.3,   # Terrasse moyenne à basse
    'Fy':       1.4,   # Basse terrasse (0-10m)
    'Fz':       1.5,   # Alluvions récentes (limons, tourbes)
    'K/Fx-y':   1.6,   # Colluvions + alluvions remaniées
    'F':        1.2,   # Zone gravière (hors eau)

    # --- DÉPÔTS DE PENTE / COLLUVIONS ---
    'CE':       2.0,   # Colluvions polygéniques, éboulis
    'CF':       2.0,   # Colluvions versant et fond de vallon
    'GZ':       1.8,   # Grèzes litées (cailloutis calcaire)
    'RFv/g1CB': 2.5,   # Formation alluviale résiduelle sur calcaire

    # --- SABLES ET GRÈS (modéré, ~2.0-2.5 ppm) ---
    'D':        2.0,   # Sables éoliens, dunes quaternaires
    'PL':       2.2,   # Sables de Lozère, Sables de Sologne
    'CPL':      2.2,   # Sables de Lozère colluvionnés
    'P-IVGC':   2.3,   # Formation détritique (gravier culminant)
    'e4S':      2.3,   # Sables fins et argiles plastiques
    'e4SB':     2.2,   # Sables grossiers de Brannay
    'e4SC-AH':  2.3,   # Sables de Cuise et supérieur
    'e4SG':     2.5,   # Sables et grès du Breuillet (arkose)
    'e4GQ':     2.5,   # Grès grossiers quartzitiques
    'e6SB-A':   2.2,   # Sables de Beauchamp et Auvers
    'e6SM':     2.0,   # Sables de Monceau
    'e6SMf':    2.0,   # Sables de Mortefontaine
    'e6MOD':    2.2,   # Sables de Monceau + Calcaire St-Ouen
    'g1FH':     2.3,   # Sables et grès de Fontainebleau
    'g1GF':     2.8,   # Grès de Fontainebleau (grésification)
    'g1SF':     2.2,   # Sables de Fontainebleau (versant)
    'g1SP':     2.3,   # Sables à galets de silex, poudingues

    # --- CALCAIRES (faible à modéré, ~2.0-2.5 ppm) ---
    'e2Cr-BE':  2.2,   # Calcaires grumeleux, pisolithiques
    'e3C':      2.0,   # Poudingue de Coye
    'e4AM':     2.0,   # Conglomérat de Meudon
    'e4PP':     2.2,   # Poudingue à chailles
    'e5-7':     2.3,   # Calcaires + marnes sableuses
    'e5C':      2.2,   # Calcaires marins indifférenciés
    'e5CG':     2.0,   # Calcaire grossier à glauconie
    'e5CL':     2.2,   # Calcaires lacustres lutétiens
    'e5MC':     2.8,   # Marnes et caillasses
    'e6':       2.2,   # Calcaire de Noisy-le-Sec
    'e6-7CH-SO':2.2,   # Calcaires de Champigny + St-Ouen
    'e6C':      2.2,   # Calcaires lagunaires bartoniens
    'e6CSO':    2.3,   # Calcaire de Saint-Ouen
    'e7C':      2.3,   # Calcaire de Champigny
    'e7CCh-MP': 2.5,   # Calcaire Champigny + Marnes
    'e7CChSi':  3.0,   # Calcaire de Champigny silicifié ← U concentré par silicification
    'g1BS':     2.3,   # Calcaire de Brie et Sannois
    'g1CB':     2.5,   # Calcaire de Brie + meulières
    'g1CCb':    2.2,   # Calcaire marin à Cardita
    'g1CD':     2.3,   # Calcaire de Darvault
    'g1CE':     2.8,   # Calcaire d'Etampes + meulières ← meulières concentrent U
    'g1CP':     2.2,   # Calcaire de Préaux
    'g1c':      2.5,   # Calcaire de Beauce inférieur + Etampes
    'g1SA':     2.5,   # Calcaire de Sannois + Argile verte
    'g1SO':     2.3,   # Calcaire de Sannois + Caillasse
    'm1CPi':    2.3,   # Calcaire de Beauce, Calcaire de Pithiviers

    # --- MARNES ET GYPSE (modéré, ~2.5-3.5 ppm) ---
    'e6-7MGC':  3.0,   # Marnes à Pholadomya, gypse (4e masse)
    'e7G':      2.5,   # Masses et marnes du gypse
    'e7G-CCh':  2.8,   # Gypse + Calcaire de Champigny
    'e7G-MP':   2.8,   # Gypse + Marnes à Pholadomya
    'e7MC':     3.0,   # Marnes ludiennes
    'e7MP':     3.0,   # Marnes à Pholadomya ludensis
    'e7MS':     3.2,   # Marnes supragypseuses (Pantin, Argenteuil)
    'g1ME':     3.0,   # Faciès marneux Calcaire d'Etampes
    'g1MH':     3.2,   # Marnes à huîtres et Argile à Corbules
    'm1MG':     3.0,   # Molasse du Gâtinais, Marnes vertes

    # --- ARGILES (élevé, ~3.5-4.5 ppm — concentratrices d'U) ---
    'e4AP':     4.0,   # Argile plastique sparnacienne ← fort
    'e4APS':    4.0,   # Argile plastique + sables
    'e4AS':     3.8,   # Argile sableuse
    'e4GA':     4.0,   # Fausses glaises + Argiles bariolées Vexin
    'e4GS':     3.8,   # Fausses glaises Vexin + Sables d'Auteuil
    'e7-g1AV':  3.5,   # Argile verte + Marnes supragypseuses
    'g1AR':     3.5,   # Argile verte de Romainville
    'g1MM':     3.5,   # Meulière et/ou Argile de Montmorency
    'p-IVMM':   3.8,   # Argile à meulière (altération silicifiée) ← fort
    'Rc':       4.0,   # Argiles à silex ← très bon concentrateur d'U

    # --- LIMONS / LOESS ---
    'LP':       2.8,   # Limon des plateaux
    'OE C':     2.5,   # Loess calcaire
    'OE L':     3.0,   # Limon loessique

    # --- DIVERS ---
    'T':        2.5,   # Zones tourbeuses
    'UCM':      2.0,   # Tuf / Travertin de La Celle
    'X':        2.0,   # Dépôts anthropiques (valeur neutre)
    'Hydro':    0.0,   # Eau (pas de contribution radon)
    'æ':        2.5,   # Grison, alios (sable ferrugineux)
}

# ============================================================
# APPLICATION SUR VOTRE COUCHE GÉOLOGIQUE
# ============================================================

bd_geol['teneur_U_ppm'] = bd_geol['NOTATION'].map(URANIUM_PPM)

# Vérification : y a-t-il des NOTATION non mappées ?
non_mappes = bd_geol[bd_geol['teneur_U_ppm'].isna()]['NOTATION'].unique()
if len(non_mappes) > 0:
    print(f"⚠️  NOTATION non mappées : {non_mappes}")
    bd_geol['teneur_U_ppm'].fillna(2.0, inplace=True)  # valeur neutre par défaut
else:
    print("✅ Toutes les formations sont mappées")

# Résumé
print("\n=== DISTRIBUTION URANIUM PAR FORMATION ===")
resume = bd_geol.groupby(['NOTATION', 'DESCR'])['teneur_U_ppm'] \
                .first().sort_values(ascending=False)
print(resume.to_string())

✅ Toutes les formations sont mappées

=== DISTRIBUTION URANIUM PAR FORMATION ===
NOTATION   DESCR                                                                                                                              
e4APS      Argile plastique, argile sableuse et Sables de Breuillet                                                                               4.0
Rc         Argiles à silex (Tertiaire à actuel)                                                                                                   4.0
e4AP       Argile plastique, sables et grès                                                                                                       4.0
e4GA       Fausses glaises, Argiles plastiques bariolées du Vexin et Sables du Soisonnais                                                         4.0
e4GS       Fausses glaises du Vexin et Sables d'Auteuil                                                                                           3.8
e4AS       Argile sableuse

In [8]:
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point

# ── 1. Chargement du fichier patients ──────────────────────────────────────
df = pd.read_csv(r"H:\PFE Loice\Notebooks\Loice_Canc-air_2025\loice_pneumodetect\Data\patients_geocoded_clean_idf_2018_2023.csv", sep=',')

gdf_patients = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df['x'], df['y']),
    crs="EPSG:4326" 
)

# ── 2. Reprojection en Lambert 93 pour correspondre à la BD Géol ───────────

gdf_patients = gdf_patients.to_crs("EPSG:2154")

# ── 3. Jointure spatiale : chaque point → polygone géologique ─────────────

gdf_joined = gpd.sjoin(
    gdf_patients[['pseudo_provisoire', 'x', 'y', 'geometry']],
    bd_geol[['NOTATION', 'DESCR', 'teneur_U_ppm', 'geometry']],
    how='left',
    predicate='within'
)

# ── 4. Gestion des patients hors polygone ─────────────────────────────────
# Quelques points peuvent tomber hors des polygones (bord de carte,
# imprécision de géocodage). On les récupère avec le polygone le plus proche.
manquants = gdf_joined[gdf_joined['NOTATION'].isna()]
print(f"Patients sans polygone trouvé : {len(manquants)}")

if len(manquants) > 0:
    gdf_repechage = gpd.sjoin_nearest(
        gdf_patients.loc[manquants.index, ['pseudo_provisoire', 'x', 'y', 'geometry']],
        bd_geol[['NOTATION', 'DESCR', 'teneur_U_ppm', 'geometry']],
        how='left'
    )
    # On remplace les lignes manquantes dans le résultat principal
    gdf_joined.loc[manquants.index, ['NOTATION', 'DESCR', 'teneur_U_ppm']] = \
        gdf_repechage[['NOTATION', 'DESCR', 'teneur_U_ppm']].values

# ── 5. Export propre : uniquement les colonnes demandées ──────────────────
resultat = gdf_joined[['pseudo_provisoire', 'x', 'y', 'NOTATION', 'DESCR', 'teneur_U_ppm']].copy()

# Vérification finale
print(f"\nTotal patients : {len(resultat)}")
print(f"Avec lithologie assignée : {resultat['NOTATION'].notna().sum()}")
print(f"Sans lithologie (à vérifier) : {resultat['NOTATION'].isna().sum()}")
print("\nAperçu :")
print(resultat.head(10).to_string(index=False))

# Sauvegarde
resultat.to_csv(r"H:\PFE Loice\Notebooks\Loice_Canc-air_2025\loice_pneumodetect\Data\patients_radon_lithologie.csv", sep=';', index=False)
print("\n✅ Fichier sauvegardé : patients_radon_lithologie.csv")

Patients sans polygone trouvé : 0

Total patients : 1682
Avec lithologie assignée : 1682
Sans lithologie (à vérifier) : 0

Aperçu :
 pseudo_provisoire        x         y NOTATION                                                                                                      DESCR  teneur_U_ppm
                16 2.572976 48.837106       LP                                                                                         Limon des plateaux           2.8
                18 2.341922 48.819523     e5MC                                                                                       Marnes et caillasses           2.8
                20 2.644045 48.540411     g1CB                                    Calcaire de Brie stampien et meulières plio-quaternaire indifférenciées           2.5
                21 2.433836 48.915017  e6-7MGC                                          Marnes à Pholadomya ludensis, Formation du gypse, Quatrième masse           3.0
                22 2.332891 

In [10]:
from sklearn.preprocessing import MinMaxScaler

# ── Chargement du résultat de la jointure ─────────────────────────────────
resultat = pd.read_csv(r"H:\PFE Loice\Notebooks\Loice_Canc-air_2025\loice_pneumodetect\Data\patients_radon_lithologie.csv", sep=';')

# ── Construction du score radon normalisé 0-100 ───────────────────────────
# On normalise la teneur_U_ppm sur l'échelle 0-100 en se basant sur
# les valeurs min/max OBSERVÉES dans votre cohorte (pas des valeurs théoriques).
# Cela garantit que le score est interprétable relativement à votre population.

scaler = MinMaxScaler(feature_range=(0, 100))
resultat['radon_score'] = scaler.fit_transform(resultat[['teneur_U_ppm']])

# ── Ajout du potentiel communal IRSN comme facteur correctif ──────────────
# Le score géologique seul ne capte pas tout : deux patients sur la même
# formation peuvent avoir des expositions très différentes selon que leur
# commune est classée à potentiel 1, 2 ou 3 par l'IRSN. On intègre donc
# cette information comme multiplicateur (et non comme remplacement).
#
# IMPORTANT : remplacez 'potentiel_radon' par le nom réel de la colonne
# dans votre fichier CSV si vous avez déjà joint les données IRSN.
# Si vous ne les avez pas encore, passez directement à l'étape suivante.

if 'potentiel_radon' in resultat.columns:
    # Convertit la classe IRSN (1/2/3) en facteur multiplicatif léger
    facteur_irsn = {1: 0.85, 2: 1.00, 3: 1.20}
    resultat['facteur_irsn'] = resultat['potentiel_radon'].map(facteur_irsn)
    resultat['radon_score'] = (resultat['radon_score'] * resultat['facteur_irsn']).clip(0, 100)
    print("✅ Score ajusté par le potentiel communal IRSN")
else:
    print("ℹ️  Score basé sur la géologie seule (pas de correction IRSN)")

# ── Statistiques descriptives du score final ──────────────────────────────
print("\n=== DISTRIBUTION DU SCORE RADON ===")
print(resultat['radon_score'].describe().round(2))

print("\n=== SCORE MOYEN PAR FORMATION GÉOLOGIQUE ===")
print(
    resultat.groupby(['NOTATION', 'DESCR'])['radon_score']
    .agg(['mean', 'count'])
    .sort_values('mean', ascending=False)
    .round(1)
    .to_string()
)

# ── Export final ──────────────────────────────────────────────────────────
resultat.to_csv(r"H:\PFE Loice\Notebooks\Loice_Canc-air_2025\loice_pneumodetect\Data\patients_radon_score_final.csv", index=False)
print("\n✅ Fichier final sauvegardé : patients_radon_score_final.csv")

ℹ️  Score basé sur la géologie seule (pas de correction IRSN)

=== DISTRIBUTION DU SCORE RADON ===
count    1682.00
mean       40.74
std        20.90
min         0.00
25%        25.93
50%        33.33
75%        55.56
max       100.00
Name: radon_score, dtype: float64

=== SCORE MOYEN PAR FORMATION GÉOLOGIQUE ===
                                                                                                                                              mean  count
NOTATION DESCR                                                                                                                                           
e4GA     Fausses glaises, Argiles plastiques bariolées du Vexin et Sables du Soisonnais                                                      100.0      9
e4APS    Argile plastique, argile sableuse et Sables de Breuillet                                                                            100.0      1
e4AP     Argile plastique, sables et grès                            